In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras import Input
import seaborn as sns
from xgboost import XGBRegressor
from prophet import Prophet
from sklearn.linear_model import LinearRegression

ModuleNotFoundError: No module named 'sklearn'

In [2]:
df = pd.read_parquet("/content/all_features_with_mortgage.parquet")
df

FileNotFoundError: [Errno 2] No such file or directory: '/content/all_features_with_mortgage.parquet'

In [ ]:
df.columns

Index(['RegionID', 'RegionName', 'StateName', 'Date', 'HomeValue',
       'IncomeNeeded', 'Inventory', 'DaysToPending', 'RentValue',
       'RenterIncomeNeeded', 'MarketHeatIndex', 'SalesCount',
       'NewConstruction', 'HomeValue_lag1', 'Inventory_lag1', 'RentValue_lag1',
       'MarketHeatIndex_lag1', 'HomeValue_roll3', 'Inventory_roll3',
       'RentValue_roll3', 'MarketHeatIndex_roll3', 'cpi', 'unrate',
       'mortgage_rate', 'Month', 'Month_Sin', 'Month_Cos',
       'HomeValue_to_Income', 'MortgageBurden', 'Inventory_to_Sales',
       'HomeValue_Change', 'IncomeNeeded_Change', 'CPI_Change',
       'MortgageRate_Change', 'Unemployment_Change', 'cpi_lag1', 'cpi_roll3',
       'mortgage_rate_lag1', 'mortgage_rate_roll3', 'unrate_lag1',
       'unrate_roll3', 'MortgageValue'],
      dtype='object')

In [ ]:
df = df.drop(['IncomeNeeded', 'RenterIncomeNeeded'], axis=1)
df

,RegionID,RegionName,StateName,Date,HomeValue,Inventory,DaysToPending,RentValue,MarketHeatIndex,SalesCount,NewConstruction,MortgageValue,HomeValue_lag1,Inventory_lag1,RentValue_lag1,MarketHeatIndex_lag1,HomeValue_roll3,Inventory_roll3,RentValue_roll3,MarketHeatIndex_roll3
0,394304,"Akron, OH",OH,2018-05-31,141222.614113,3632.0,58.0,810.850885,44.0,1071.0,79.0,578.230936,140534.856845,3385.0,811.126946,43.0,140507.606961,3405.000000,810.611908,42.666667
1,394304,"Akron, OH",OH,2018-06-30,141720.548498,3806.0,55.0,812.803919,41.0,967.0,68.0,579.186971,141222.614113,3632.0,810.850885,44.0,141159.339818,3607.666667,811.593916,42.666667
2,394304,"Akron, OH",OH,2018-07-31,142231.927700,3917.0,54.0,812.604899,40.0,1045.0,71.0,578.395359,141720.548498,3806.0,812.803919,41.0,141725.030104,3785.000000,812.086568,41.666667
3,394304,"Akron, OH",OH,2018-08-31,142828.303623,3968.0,53.0,814.831361,37.0,1023.0,71.0,582.351584,142231.927700,3917.0,812.604899,40.0,142260.259941,3897.000000,813.413393,39.333333
4,394304,"Akron, OH",OH,2018-09-30,143564.616358,3918.0,56.0,817.904047,34.0,802.0,57.0,590.669795,142828.303623,3968.0,814.831361,37.0,142874.949227,3934.333333,815.113436,37.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8091,395238,"Worcester, MA",MA,2025-04-30,468584.529291,1495.0,26.0,2093.804717,78.0,741.0,41.0,2425.157969,468612.638793,1315.0,2087.114145,82.0,468510.555487,1355.333333,2086.006474,81.666667
8092,395238,"Worcester, MA",MA,2025-05-31,468027.404006,1799.0,21.0,2098.248862,71.0,905.0,59.0,2444.942673,468584.529291,1495.0,2093.804717,78.0,468408.190696,1536.333333,2093.055908,77.000000
8093,395238,"Worcester, MA",MA,2025-06-30,467416.502701,2037.0,20.0,2107.961690,68.0,1090.0,54.0,2442.125250,468027.404006,1799.0,2098.248862,71.0,468009.478666,1777.000000,2100.005090,72.333333
8094,395238,"Worcester, MA",MA,2025-07-31,467202.194185,2174.0,21.0,2115.546334,67.0,1038.0,39.0,2416.762894,467416.502701,2037.0,2107.961690,68.0,467548.700297,2003.333333,2107.252295,68.666667


In [ ]:
df.columns

Index(['RegionID', 'RegionName', 'StateName', 'Date', 'HomeValue', 'Inventory',
       'DaysToPending', 'RentValue', 'MarketHeatIndex', 'SalesCount',
       'NewConstruction', 'MortgageValue', 'HomeValue_lag1', 'Inventory_lag1',
       'RentValue_lag1', 'MarketHeatIndex_lag1', 'HomeValue_roll3',
       'Inventory_roll3', 'RentValue_roll3', 'MarketHeatIndex_roll3'],
      dtype='object')

In [ ]:
treasury = pd.read_csv("/content/GS10.csv")
fed_funds = pd.read_csv("/content/FEDFUNDS.csv")
personal_income = pd.read_csv("/content/DSPIC96.csv")

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
treasury["observation_date"] = pd.to_datetime(treasury["observation_date"])
fed_funds["observation_date"] = pd.to_datetime(fed_funds["observation_date"])
personal_income["observation_date"] = pd.to_datetime(personal_income["observation_date"])

In [ ]:
treasury.rename(columns={"observation_date":"Date", "GS10": "treasury_yield"}, inplace=True)
fed_funds.rename(columns={"observation_date":"Date", "FEDFUNDS": "funds_rate"}, inplace=True)
personal_income.rename(columns={"observation_date":"Date", "DSPIC96": "personal_income"}, inplace=True)

In [ ]:
df = pd.merge(
    df,
    treasury,
    on='Date',
    how='left'
)
df = pd.merge(
    df,
    fed_funds,
    on='Date',
    how='left'
)
df = pd.merge(
    df,
    personal_income,
    on='Date',
    how='left'
)
df

,RegionID,RegionName,StateName,Date,HomeValue,Inventory,DaysToPending,RentValue,MarketHeatIndex,SalesCount,...,Unemployment_Change,cpi_lag1,cpi_roll3,mortgage_rate_lag1,mortgage_rate_roll3,unrate_lag1,unrate_roll3,treasury_yield,funds_rate,personal_income
0,394304,"Akron, OH",OH,2018-05-01,141498.445820,3632.0,58.0,842.862503,44.0,1071.0,...,0.000000,0.000,0.000000,0.0000,0.000000,0.0,0.000000,2.98,1.70,15057.8
1,394304,"Akron, OH",OH,2018-06-01,141997.352756,3806.0,55.0,844.909823,41.0,967.0,...,0.052632,250.792,0.000000,4.5860,0.000000,3.8,0.000000,2.91,1.82,15119.3
2,394304,"Akron, OH",OH,2018-07-01,142509.730768,3917.0,54.0,844.722835,40.0,1045.0,...,-0.050000,251.018,251.008000,4.5700,4.561167,4.0,3.866667,2.89,1.91,15184.7
3,394304,"Akron, OH",OH,2018-08-01,143107.271515,3968.0,53.0,846.884637,37.0,1023.0,...,0.000000,251.214,251.298333,4.5275,4.549167,3.8,3.866667,2.89,1.91,15238.6
4,394304,"Akron, OH",OH,2018-09-01,143845.022393,3918.0,56.0,850.004092,34.0,803.0,...,-0.026316,251.663,251.686333,4.5500,4.568333,3.8,3.766667,3.00,1.95,15237.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999,395238,"Worcester, MA",MA,2025-03-01,471121.892980,1315.0,33.0,2118.639336,82.0,607.0,...,0.024390,319.775,319.492000,6.8425,6.816833,4.1,4.100000,4.28,4.33,18029.1
8000,395238,"Worcester, MA",MA,2025-04-01,471093.632963,1495.0,26.0,2125.944172,78.0,742.0,...,0.000000,319.615,319.903667,6.6500,6.739167,4.2,4.166667,4.28,4.33,18168.6
8001,395238,"Worcester, MA",MA,2025-05-01,470533.524469,1799.0,21.0,2130.885088,71.0,903.0,...,0.000000,320.321,320.172000,6.7250,6.730333,4.2,4.200000,4.42,4.33,18041.7
8002,395238,"Worcester, MA",MA,2025-06-01,469919.352005,2037.0,20.0,2140.907753,68.0,1089.0,...,-0.023810,320.580,320.800333,6.8160,6.786167,4.2,4.166667,4.38,4.33,18036.2


In [ ]:
df.columns

Index(['RegionID', 'RegionName', 'StateName', 'Date', 'HomeValue', 'Inventory',
       'DaysToPending', 'RentValue', 'MarketHeatIndex', 'SalesCount',
       'NewConstruction', 'HomeValue_lag1', 'Inventory_lag1', 'RentValue_lag1',
       'MarketHeatIndex_lag1', 'HomeValue_roll3', 'Inventory_roll3',
       'RentValue_roll3', 'MarketHeatIndex_roll3', 'cpi', 'unrate',
       'mortgage_rate', 'Month', 'Month_Sin', 'Month_Cos', 'MortgageBurden',
       'Inventory_to_Sales', 'HomeValue_Change', 'CPI_Change',
       'MortgageRate_Change', 'Unemployment_Change', 'cpi_lag1', 'cpi_roll3',
       'mortgage_rate_lag1', 'mortgage_rate_roll3', 'unrate_lag1',
       'unrate_roll3', 'treasury_yield', 'funds_rate', 'personal_income'],
      dtype='object')

In [ ]:
def evaluate_metro(df, target, model, features, params=None, test_months=12 ):
  train = df.iloc[:-test_months]
  test  = df.iloc[-test_months:]
  X_train, y_train = train[features], train[target]
  X_test,  y_test  = test[features],  test[target]
  evaluate_model = model(**params) if params else model()
  evaluate_model.fit(X_train, y_train)
  pred = evaluate_model.predict(X_test)
  mae  = mean_absolute_error(y_test, pred)
  rmse = np.sqrt(mean_squared_error(y_test, pred))
  return mae, rmse

In [ ]:
linear_features = [
    'HomeValue', 'Inventory', 'SalesCount', 'DaysToPending', 'NewConstruction',
    'MarketHeatIndex', 'cpi', 'unrate', 'mortgage_rate', 'treasury_yield',
    'funds_rate', 'personal_income', 'MortgageBurden', 'Inventory_to_Sales',
    'HomeValue_Change', 'CPI_Change', 'MortgageRate_Change', 'Unemployment_Change',
    'Month_Sin', 'Month_Cos',
    'HomeValue_lag1', 'Inventory_lag1', 'MarketHeatIndex_lag1'
]

results = []

for region, df_region in df.groupby("RegionID"):
  if len(df_region) > 15:
    MODEL_CLASS = LinearRegression
    mae, rmse = evaluate_metro(df_region, "RentValue", MODEL_CLASS, linear_features)
    results.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "MAE": mae,
        "RMSE": rmse,
    })

results_df = pd.DataFrame(results)

In [ ]:
results_df[['MAE','RMSE']].mean()

,0
MAE,28.885193
RMSE,32.693147


In [ ]:
#Prophet
regressors = [
    'cpi', 'unrate', 'treasury_yield', 'funds_rate', 'personal_income'
]

def evaluate_metro_prophet(df, target,  test_months=12):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df = df.rename(columns={"Date": "ds", target: "y"}).dropna(subset=["y"])
    numeric_cols = [c for c in numeric_cols if c != target]
    df = df.set_index("ds").resample("ME")[numeric_cols + ["y"]].mean().reset_index()
    train = df.iloc[:-test_months]
    test  = df.iloc[-test_months:]

    model = Prophet(yearly_seasonality=True)
    for reg in regressors:
        model.add_regressor(reg)

    model.fit(train[["ds", "y"] + regressors])

    future = model.make_future_dataframe(periods=test_months, freq="ME")
    future = future.merge(df[["ds"] + regressors], on="ds", how="left")

    forecast = model.predict(future)

    y_pred = forecast["yhat"][-test_months:].values
    y_true = test["y"].values

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {"MAE": mae, "RMSE": rmse, "Forecast": forecast}

In [ ]:
prophet_results = []

for region, df_region in df.groupby("RegionID"):
    if len(df_region) < 24:
        continue
    results = evaluate_metro_prophet(df_region, "RentValue")
    prophet_results.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "MAE_prophet": results["MAE"],
        "RMSE_prophet": results["RMSE"],
        "Forecast": results["Forecast"]
    })

prophet_results_df = pd.DataFrame(prophet_results)

INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmpj7a91z3x/uq92_zhg.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpj7a91z3x/m4e_nfwe.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=18802', 'data', 'file=/tmp/tmpj7a91z3x/uq92_zhg.json', 'init=/tmp/tmpj7a91z3x/m4e_nfwe.json', 'output', 'file=/tmp/tmpj7a91z3x/prophet_modelgnt65pkk/prophet_model-20251017222258.csv', 'method=optimize', 'algorithm=newton', 'iter=10000']
22:22:58 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
22:23:00 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
INFO:prophet:Disabling weekly seasonali

In [ ]:
prophet_results_df[["MAE_prophet", "RMSE_prophet"]].mean()

,0
MAE_prophet,12.696555
RMSE_prophet,14.460824


In [ ]:
linear_features_mortgage = [
    'HomeValue', 'Inventory', 'SalesCount', 'DaysToPending', 'NewConstruction',
    'MarketHeatIndex', 'cpi', 'unrate', 'treasury_yield', 'funds_rate',
    'personal_income', 'MortgageBurden', 'Inventory_to_Sales',
    'HomeValue_Change', 'CPI_Change', 'Unemployment_Change',
    'Month_Sin', 'Month_Cos',
    'HomeValue_lag1', 'Inventory_lag1', 'MarketHeatIndex_lag1'
]

results_mortgage = []

for region, df_region in df.groupby("RegionID"):
  if len(df_region) > 15:
    MODEL_CLASS = LinearRegression
    mae, rmse = evaluate_metro(df_region, "mortgage_rate", MODEL_CLASS, linear_features)
    results_mortgage.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "MAE": mae,
        "RMSE": rmse,
    })

results_mortgage_df = pd.DataFrame(results_mortgage)

In [ ]:
results_mortgage_df[['MAE','RMSE']].mean()

,0
MAE,7.566492e-14
RMSE,7.677804e-14


In [ ]:
prophet_results_mortgage = []

for region, df_region in df.groupby("RegionID"):
    if len(df_region) < 24:
        continue
    results = evaluate_metro_prophet(df_region, "mortgage_rate")
    prophet_results_mortgage.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "MAE_prophet": results["MAE"],
        "RMSE_prophet": results["RMSE"],
        "Forecast": results["Forecast"]
    })
prophet_mortgage_results_df = pd.DataFrame(prophet_results_mortgage)

INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmpj7a91z3x/uckyfd53.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpj7a91z3x/bmep6qx4.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=52851', 'data', 'file=/tmp/tmpj7a91z3x/uckyfd53.json', 'init=/tmp/tmpj7a91z3x/bmep6qx4.json', 'output', 'file=/tmp/tmpj7a91z3x/prophet_modeleso3pj4_/prophet_model-20251017223816.csv', 'method=optimize', 'algorithm=newton', 'iter=10000']
22:38:16 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
22:38:16 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
INFO:prophet:Disabling weekly seasonali

In [ ]:
prophet_mortgage_results_df[["MAE_prophet", "RMSE_prophet"]].mean()

,0
MAE_prophet,0.200900
RMSE_prophet,0.240986


In [ ]:
df["mortgage_rate"]

,mortgage_rate
0,4.5860
1,4.5700
2,4.5275
3,4.5500
4,4.6275
...,...
7999,6.6500
8000,6.7250
8001,6.8160
8002,6.8175
